# Student Wellbeing Statistical Analysis & Probability – Day 16

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("Day16_Student_Wellbeing_Survey.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
display(df.head())

## 3. Mean, Median and Mode

Mean = Σx / n

Median = middle value after sorting

Mode = most frequent value

In [ ]:
variables = [
    "Weekly_Study_Hours",
    "Average_Sleep_Hours",
    "Daily_Screen_Time_Hours",
    "Stress_Score",
    "Academic_Readiness_Score"
]

central_tendency = pd.DataFrame(index=variables)
central_tendency["Mean"] = df[variables].mean()
central_tendency["Median"] = df[variables].median()
central_tendency["Mode"] = [df[col].mode().iloc[0] for col in variables]

display(central_tendency)

## 4. Range, Variance, Standard Deviation, Q1, Q3 and IQR

Range = Maximum − Minimum

IQR = Q3 − Q1

In [ ]:
summary = pd.DataFrame(index=variables)
summary["Range"] = df[variables].max() - df[variables].min()
summary["Variance"] = df[variables].var()
summary["Standard_Deviation"] = df[variables].std()
summary["Q1"] = df[variables].quantile(0.25)
summary["Q3"] = df[variables].quantile(0.75)
summary["IQR"] = summary["Q3"] - summary["Q1"]

display(summary)

greatest = summary["Standard_Deviation"].idxmax()
print("Greatest variability by standard deviation:", greatest)

## 5. IQR Outlier Detection

Outlier rule: x < Q1 − 1.5×IQR or x > Q3 + 1.5×IQR.

In [ ]:
outlier_variables = [
    "Weekly_Study_Hours",
    "Daily_Screen_Time_Hours",
    "Commute_Time_Minutes",
    "Monthly_Discretionary_Spending"
]

outlier_rows = []
for col in outlier_variables:
    q1 = df[col].quantile(.25)
    q3 = df[col].quantile(.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.append([col, q1, q3, iqr, lower, upper, int(mask.sum())])

outlier_table = pd.DataFrame(
    outlier_rows,
    columns=["Variable","Q1","Q3","IQR","Lower_Bound","Upper_Bound","Outlier_Count"]
)
display(outlier_table)

## 6. Mean and Median Before and After Removing Outliers

In [ ]:
col = "Weekly_Study_Hours"
q1 = df[col].quantile(.25)
q3 = df[col].quantile(.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

filtered = df[(df[col] >= lower) & (df[col] <= upper)]

comparison = pd.DataFrame({
    "Mean": [df[col].mean(), filtered[col].mean()],
    "Median": [df[col].median(), filtered[col].median()],
    "Records": [len(df), len(filtered)]
}, index=["Before", "After"])

display(comparison)

## 7. Define Probability Events

A = Part_Time_Job is Yes

B = Stress_Score ≥ 7

C = Scholarship is Yes

D = Exercises at least 3 days per week

In [ ]:
A = df["Part_Time_Job"].astype(str).str.strip().str.lower().eq("yes")
B = df["Stress_Score"] >= 7
C = df["Scholarship"].astype(str).str.strip().str.lower().eq("yes")

exercise_values = pd.to_numeric(
    df["Exercise_Days_Per_Week"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)
D = exercise_values >= 3

print("Exercise column:", "Exercise_Days_Per_Week")
print("Unique exercise values:", df["Exercise_Days_Per_Week"].unique())

## 8. Required Probabilities

P(E) = favorable outcomes / total outcomes

P(A or B) = P(A) + P(B) − P(A and B)

P(A|B) = P(A and B) / P(B)

In [ ]:
P_A = A.mean()
P_B = B.mean()
P_C = C.mean()
P_D = D.mean()
P_A_or_B = (A | B).mean()
P_A_and_B = (A & B).mean()
P_A_given_B = (A & B).sum() / B.sum()
P_B_given_A = (A & B).sum() / A.sum()

probabilities = pd.Series({
    "P(A)": P_A,
    "P(B)": P_B,
    "P(C)": P_C,
    "P(D)": P_D,
    "P(A or B)": P_A_or_B,
    "P(A and B)": P_A_and_B,
    "P(A | B)": P_A_given_B,
    "P(B | A)": P_B_given_A
})
display(probabilities.to_frame("Probability"))

## 9. Mutually Exclusive: Year 1 and Year 4

In [ ]:
year1 = df["Year_of_Study"] == 1
year4 = df["Year_of_Study"] == 4
intersection = (year1 & year4).sum()

print("Students in both Year 1 and Year 4:", intersection)
print("Mutually exclusive:", intersection == 0)

## 10. Independence of A and B

If independent: P(A and B) = P(A) × P(B).

In [ ]:
product = P_A * P_B
print("P(A and B) =", P_A_and_B)
print("P(A) × P(B) =", product)
print("Difference =", P_A_and_B - product)

if np.isclose(P_A_and_B, product):
    print("Conclusion: A and B appear independent.")
else:
    print("Conclusion: A and B do not appear independent.")

## 11. Bayes' Theorem

P(A|B) = [P(B|A) × P(A)] / P(B)

Also calculate P(B|not A).

In [ ]:
P_not_A = 1 - P_A
P_B_given_not_A = ((~A) & B).sum() / (~A).sum()

P_B_total = P_B_given_A * P_A + P_B_given_not_A * P_not_A
P_A_given_B_bayes = P_B_given_A * P_A / P_B_total

bayes = pd.Series({
    "P(A)": P_A,
    "P(B|A)": P_B_given_A,
    "P(B|not A)": P_B_given_not_A,
    "P(B) from total probability": P_B_total,
    "P(A|B) direct": P_A_given_B,
    "P(A|B) by Bayes": P_A_given_B_bayes
})
display(bayes.to_frame("Value"))
print("Direct and Bayes agree:", np.isclose(P_A_given_B, P_A_given_B_bayes))

## 12. Normal Distribution: Academic Readiness Score

In [ ]:
scores = df["Academic_Readiness_Score"].dropna()
mu = scores.mean()
sigma = scores.std()

high = scores.max()
low = scores.min()

z_high = (high - mu) / sigma
z_low = (low - mu) / sigma

print("Mean:", mu)
print("Standard deviation:", sigma)
print("Highest score:", high)
print("Highest score Z:", z_high)
print("Lowest score:", low)
print("Lowest score Z:", z_low)

plt.figure(figsize=(9,5))
sns.histplot(scores, bins=20, kde=True)
plt.axvline(mu, linestyle="--", label="Mean")
plt.title("Academic Readiness Score Distribution")
plt.xlabel("Academic Readiness Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

**Interpretation:** A positive Z-score is above the mean and a negative Z-score is below the mean. The absolute value indicates the number of standard deviations from the mean.

## 13. 68–95–99.7 Empirical Rule

In [ ]:
empirical_rule = pd.DataFrame({
    "Interval": ["Within ±1 SD", "Within ±2 SD", "Within ±3 SD"],
    "Expected Percentage": ["68%", "95%", "99.7%"]
})
display(empirical_rule)

**Interpretation:** For an approximately normal Academic Readiness Score distribution, about 68%, 95%, and 99.7% of students are expected within 1, 2, and 3 standard deviations of the mean respectively.

## 14. Five Meaningful Statistical Observations

In [ ]:
print("1. Greatest variability:", greatest)
print("2. Weekly Study Hours outliers:", int(((df["Weekly_Study_Hours"] < lower) | (df["Weekly_Study_Hours"] > upper)).sum()))
print("3. P(A) =", round(P_A, 4), "and P(B) =", round(P_B, 4))
print("4. P(A and B) =", round(P_A_and_B, 4), "versus P(A)×P(B) =", round(product, 4))
print("5. Academic Readiness mean =", round(mu, 2), "and standard deviation =", round(sigma, 2))

## Conclusion

The notebook completes the required descriptive statistics, IQR outlier analysis, probability calculations, mutual-exclusivity test, independence check, Bayes' theorem verification, and normal-distribution analysis.